In [1]:
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import pandas as pd

In [2]:
# ------------------- Paths -------------------
RUN_DIR = Path("../../2_final_runs/3_uy_basins/runs")
hydrographs_dir = Path("./hydrographs")
metrics_dir = Path("/home/azureuser/NeuralHydrologyAzure/2_final_runs/3_uy_basins/ensemble_testing_metrics")

In [3]:
# run_patterns = {
#     "CARAVAN": "precip_prcp_mm_day_seed_*",
#     "CHIRPS":  "precip_prcp_chirps_mm_day_seed_*",
#     "MSWEP":   "precip_prcp_mswep_mm_day_seed_*",
#     "GAUGES":  "precip_prcp_gauge_mm_day_seed_*",
# }

run_patterns = {
    "CARAVAN": "precip_prcp_mm_day_seed_*",
    "CHIRPS":  "precip_prcp_chirps_mm_day_seed_*",
    "MSWEP":   "precip_prcp_mswep_mm_day_seed_*",
    "GAUGES":  "precip_prcp_gauge_mm_day_seed_*",
    "CARAVAN_CHIRPS":  "precip_prcp_mm_day_prcp_chirps_mm_day_seed_*",
    "CARAVAN_MSWEP":  "precip_prcp_mm_day_prcp_mswep_mm_day_seed_*",
    "CARAVAN_GAUGES":  "precip_prcp_mm_day_prcp_gauge_mm_day_seed_*",
    "CHIRPS_MSWEP":  "precip_prcp_chirps_mm_day_prcp_mswep_mm_day_seed_*",
    "CHIRPS_GAUGES": "precip_prcp_chirps_mm_day_prcp_gauge_mm_day_seed_*",
    "MSWEP_GAUGES": "precip_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*",
    "CARAVAN_CHIRPS_MSWEP":  "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_seed_*",
    "CARAVAN_CHIRPS_GAUGES":  "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_gauge_mm_day_seed_*",
    "CARAVAN_MSWEP_GAUGES":  "precip_prcp_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*",
    "CHIRPS_MSWEP_GAUGES":  "precip_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*",
    "CARAVAN_CHIRPS_MSWEP_GAUGES":  "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*"
}

In [4]:
ensemble_by_pattern = {}

for label, pattern in run_patterns.items():
    matched_paths = sorted(RUN_DIR.glob(f"{pattern}/test/model_epoch030/test_results.p"))
    print(f"[{label}] Found {len(matched_paths)} runs: {[p.parts[-4] for p in matched_paths]}")
    
    if not matched_paths:
        print(f"  WARNING: No runs found for '{label}', skipping.")
        continue
    
    all_runs_data = []
    for file_path in matched_paths:
        with open(file_path, "rb") as f:
            all_runs_data.append(pickle.load(f))
    
    ensemble_data = {}
    for basin_id in all_runs_data[0].keys():
        sims = np.stack([
            run[basin_id]['1D']['xr']['QObs_mm_d_sim'].values
            for run in all_runs_data
        ], axis=0)
        
        mean_sim = np.mean(sims, axis=0)
        
        xr_ensemble = all_runs_data[0][basin_id]['1D']['xr'].copy(deep=True)
        xr_ensemble['QObs_mm_d_sim'].values[:] = mean_sim
        
        ensemble_data[basin_id] = {'1D': {'xr': xr_ensemble}}
    
    ensemble_by_pattern[label] = ensemble_data

[CARAVAN] Found 8 runs: ['precip_prcp_mm_day_seed_111_2202_184533', 'precip_prcp_mm_day_seed_222_2202_185112', 'precip_prcp_mm_day_seed_333_2202_185649', 'precip_prcp_mm_day_seed_444_2202_190226', 'precip_prcp_mm_day_seed_555_2302_064058', 'precip_prcp_mm_day_seed_666_2302_064647', 'precip_prcp_mm_day_seed_777_2302_065235', 'precip_prcp_mm_day_seed_888_2302_065823']
[CHIRPS] Found 8 runs: ['precip_prcp_chirps_mm_day_seed_111_2904_071050', 'precip_prcp_chirps_mm_day_seed_222_2904_071648', 'precip_prcp_chirps_mm_day_seed_333_2904_072228', 'precip_prcp_chirps_mm_day_seed_444_2904_072803', 'precip_prcp_chirps_mm_day_seed_555_2904_073338', 'precip_prcp_chirps_mm_day_seed_666_2904_073913', 'precip_prcp_chirps_mm_day_seed_777_2904_074449', 'precip_prcp_chirps_mm_day_seed_888_2904_075024']
[MSWEP] Found 8 runs: ['precip_prcp_mswep_mm_day_seed_111_2804_222722', 'precip_prcp_mswep_mm_day_seed_222_2804_223324', 'precip_prcp_mswep_mm_day_seed_333_2804_223907', 'precip_prcp_mswep_mm_day_seed_444_28

In [5]:
ensemble_by_pattern

{'CARAVAN': {'CAMELS_UY_10': {'1D': {'xr': <xarray.Dataset> Size: 35kB
    Dimensions:        (date: 2191, time_step: 1)
    Coordinates:
      * date           (date) datetime64[ns] 18kB 2008-10-01 ... 2014-09-30
      * time_step      (time_step) int64 8B 0
    Data variables:
        QObs_mm_d_obs  (date, time_step) float32 9kB 0.2376 0.2216 ... 1.265 1.686
        QObs_mm_d_sim  (date, time_step) float32 9kB 0.4299 0.4396 ... 1.171 1.045}},
  'CAMELS_UY_11': {'1D': {'xr': <xarray.Dataset> Size: 35kB
    Dimensions:        (date: 2191, time_step: 1)
    Coordinates:
      * date           (date) datetime64[ns] 18kB 2008-10-01 ... 2014-09-30
      * time_step      (time_step) int64 8B 0
    Data variables:
        QObs_mm_d_obs  (date, time_step) float32 9kB 0.3589 0.3589 ... 0.8041 0.6947
        QObs_mm_d_sim  (date, time_step) float32 9kB 0.8108 0.7574 ... 0.6808 0.6359}},
  'CAMELS_UY_15': {'1D': {'xr': <xarray.Dataset> Size: 35kB
    Dimensions:        (date: 2191, time_step: 1)

In [6]:
# Map labels to their metrics filename (pattern without the _*)
# metrics_filenames = {
#     "CARAVAN": "precip_prcp_mm_day.csv",
#     "CHIRPS":  "precip_prcp_chirps_mm_day.csv",
#     "MSWEP":   "precip_prcp_mswep_mm_day.csv",
#     "GAUGES":  "precip_prcp_gauge_mm_day.csv",
# }

metrics_filenames = {
    "CARAVAN": "precip_prcp_mm_day.csv",
    "CHIRPS":  "precip_prcp_chirps_mm_day.csv",
    "MSWEP":   "precip_prcp_mswep_mm_day.csv",
    "GAUGES":  "precip_prcp_gauge_mm_day.csv",
    "CARAVAN_CHIRPS":  "precip_prcp_mm_day_prcp_chirps_mm_day.csv",
    "CARAVAN_MSWEP":  "precip_prcp_mm_day_prcp_mswep_mm_day.csv",
    "CARAVAN_GAUGES":  "precip_prcp_mm_day_prcp_gauge_mm_day.csv",
    "CHIRPS_MSWEP":  "precip_prcp_chirps_mm_day_prcp_mswep_mm_day.csv",
    "CHIRPS_GAUGES": "precip_prcp_chirps_mm_day_prcp_gauge_mm_day.csv",
    "MSWEP_GAUGES": "precip_prcp_mswep_mm_day_prcp_gauge_mm_day.csv",
    "CARAVAN_CHIRPS_MSWEP":  "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day.csv",
    "CARAVAN_CHIRPS_GAUGES":  "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_gauge_mm_day.csv",
    "CARAVAN_MSWEP_GAUGES":  "precip_prcp_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day.csv",
    "CHIRPS_MSWEP_GAUGES":  "precip_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day.csv",
    "CARAVAN_CHIRPS_MSWEP_GAUGES":  "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day.csv"
}

for label, basin_dict in ensemble_by_pattern.items():
    out_dir = hydrographs_dir / label
    out_dir.mkdir(parents=True, exist_ok=True)

    metrics_df = pd.read_csv(metrics_dir / metrics_filenames[label], index_col="basin_id")

    for basin_id, data in basin_dict.items():
        xr_ds = data['1D']['xr']

        obs  = xr_ds['QObs_mm_d_obs'].values.squeeze()
        sim  = xr_ds['QObs_mm_d_sim'].values.squeeze()
        time = pd.to_datetime(xr_ds['date'].values)

        if basin_id in metrics_df.index:
            basin_nse = metrics_df.loc[basin_id, "NSE"]
            nse_str = f"{float(basin_nse):.3f}"
        else:
            nse_str = "N/A"

        plt.figure(figsize=(12, 5))
        plt.plot(time, obs, label="Observed", alpha=0.6)
        plt.plot(time, sim, label="Simulated", alpha=0.6)
        plt.xlabel("Date", fontsize=14)
        plt.ylabel("Streamflow (mm/day)", fontsize=14)
        plt.title(f"Test - {basin_id} - {label} - NSE: {nse_str}", fontsize=16)
        plt.legend(fontsize=13)
        plt.grid(True, alpha=0.35)
        plt.tick_params(axis='both', labelsize=12)
        plt.tight_layout()

        plt.savefig(out_dir / f"{basin_id}.png", dpi=150)
        plt.close()

    print(f"[{label}] Saved {len(basin_dict)} hydrographs → {out_dir}")

[CARAVAN] Saved 11 hydrographs → hydrographs/CARAVAN
[CHIRPS] Saved 11 hydrographs → hydrographs/CHIRPS
[MSWEP] Saved 11 hydrographs → hydrographs/MSWEP
[GAUGES] Saved 11 hydrographs → hydrographs/GAUGES
[CARAVAN_CHIRPS] Saved 11 hydrographs → hydrographs/CARAVAN_CHIRPS
[CARAVAN_MSWEP] Saved 11 hydrographs → hydrographs/CARAVAN_MSWEP
[CARAVAN_GAUGES] Saved 11 hydrographs → hydrographs/CARAVAN_GAUGES
[CHIRPS_MSWEP] Saved 11 hydrographs → hydrographs/CHIRPS_MSWEP
[CHIRPS_GAUGES] Saved 11 hydrographs → hydrographs/CHIRPS_GAUGES
[MSWEP_GAUGES] Saved 11 hydrographs → hydrographs/MSWEP_GAUGES
[CARAVAN_CHIRPS_MSWEP] Saved 11 hydrographs → hydrographs/CARAVAN_CHIRPS_MSWEP
[CARAVAN_CHIRPS_GAUGES] Saved 11 hydrographs → hydrographs/CARAVAN_CHIRPS_GAUGES
[CARAVAN_MSWEP_GAUGES] Saved 11 hydrographs → hydrographs/CARAVAN_MSWEP_GAUGES
[CHIRPS_MSWEP_GAUGES] Saved 11 hydrographs → hydrographs/CHIRPS_MSWEP_GAUGES
[CARAVAN_CHIRPS_MSWEP_GAUGES] Saved 11 hydrographs → hydrographs/CARAVAN_CHIRPS_MSWEP_GAUG